In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcommerceProject") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

In [2]:
# For huge data always construct schema manually. Don't use inferschema because it takes a lot of time and memory
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [3]:
# Defining Schema manually

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

In [4]:
df = spark.read.csv("2019-Oct.csv", header=True, schema=schema)
df.show()

+--------------------+----------------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|          event_time|      event_type|product_id|        category_id|category_code|   brand|price|  user_id|        user_session|
+--------------------+----------------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|2019-10-01 00:00:...|            cart|   5773203|1487580005134238553|         NULL|  runail| 2.62|463240011|26dd6e6e-4dac-477...|
|2019-10-01 00:00:...|            cart|   5773353|1487580005134238553|         NULL|  runail| 2.62|463240011|26dd6e6e-4dac-477...|
|2019-10-01 00:00:...|            cart|   5881589|2151191071051219817|         NULL|  lovely|13.48|429681830|49e8d843-adf3-428...|
|2019-10-01 00:00:...|            cart|   5723490|1487580005134238553|         NULL|  runail| 2.62|463240011|26dd6e6e-4dac-477...|
|2019-10-01 00:00:...|            cart|   5881449|1487580013522845895|         NULL

In [5]:
df.describe().show()

+-------+--------------------+----------+------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|summary|          event_time|event_type|        product_id|         category_id|      category_code|   brand|            price|            user_id|        user_session|
+-------+--------------------+----------+------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|  count|             4102283|   4102283|           4102283|             4102283|              67477| 2443022|          4102283|            4102283|             4101646|
|   mean|                NULL|      NULL| 5468463.875820367|1.545651907978867...|               NULL|    NULL|8.534920267569731|5.013416264570426E8|                NULL|
| stddev|                NULL|      NULL|1321862.7673877932|1.563666472056598...|               NULL|    NULL|19.13314897118914|8.171256905243336E7|  

In [6]:
df.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



### Now Our goal is to improve the data quality

Things which we need to improve

1. From describe method we can see the minimum value of price is in negative. And price should not be less than Zero so we will gonna find out number of rows with price problem. If the number is small than dropping the rows is feasable options
2. 

In [7]:
df.filter(df.price <= 0).count()

6086

In [8]:
# Let's create a new data frame in which rows with the price problem isn't present

df_cleaned = df.filter(df.price > 0)

In [9]:
df_cleaned.describe().show()

+-------+--------------------+----------+------------------+--------------------+-------------------+--------+------------------+-------------------+--------------------+
|summary|          event_time|event_type|        product_id|         category_id|      category_code|   brand|             price|            user_id|        user_session|
+-------+--------------------+----------+------------------+--------------------+-------------------+--------+------------------+-------------------+--------------------+
|  count|             4096197|   4096197|           4096197|             4096197|              67352| 2443016|           4096197|            4096197|             4095560|
|   mean|                NULL|      NULL| 5467833.478498471|1.545586038596165...|               NULL|    NULL| 8.547736810511498| 5.01357513612451E8|                NULL|
| stddev|                NULL|      NULL|1322743.0186315412|1.562703483593387...|               NULL|    NULL|19.144324820786924|8.17002432392017

In [10]:
# 3. Validation: Compare the counts to see how much data you removed
original_count = df.count()
cleaned_count = df_cleaned.count()
removed_rows = original_count - cleaned_count

print(f"Original Rows: {original_count}")
print(f"Cleaned Rows:  {cleaned_count}")
print(f"Rows removed due to price errors: {removed_rows}")

Original Rows: 4102283
Cleaned Rows:  4096197
Rows removed due to price errors: 6086


In [11]:
# Filling the brand column
df_cleaned = df_cleaned.fillna("Unknown_brand", subset=["brand"])

In [12]:
# Find IDs that have BOTH a name and a Null
ids_with_names = df_cleaned.filter(df_cleaned.brand.isNotNull()).select("product_id").distinct()
ids_with_nulls = df_cleaned.filter(df_cleaned.brand.isNull()).select("product_id").distinct()

# Intersect them to see if any IDs are in both lists
reconstructable_ids = ids_with_names.intersect(ids_with_nulls)

print(f"Number of products we can actually fix: {reconstructable_ids.count()}")

Number of products we can actually fix: 0


In [13]:
# Filling values in category_code
df_cleaned = df_cleaned.fillna("accessories", subset=["category_code"])

In [14]:
# If it's 'electronics.audio', getItem(0) returns 'electronics'
# If it's 'accessory', getItem(0) returns 'accessory'
from pyspark.sql import functions as F

# If it's 'electronics.audio', getItem(0) returns 'electronics'
# If it's 'accessory', getItem(0) returns 'accessory'
df_cleaned = df_cleaned.withColumn("department", F.split(F.col("category_code"), "\.").getItem(0))

In [15]:
# 3. Verify the logic
df_cleaned.select("category_code", "department").distinct().show(10)

+--------------------+-----------+
|       category_code| department|
+--------------------+-----------+
|furniture.bathroo...|  furniture|
|appliances.enviro...| appliances|
|       apparel.glove|    apparel|
|appliances.enviro...| appliances|
|         accessories|accessories|
|     accessories.bag|accessories|
|furniture.living_...|  furniture|
|accessories.cosme...|accessories|
|appliances.person...| appliances|
|furniture.living_...|  furniture|
+--------------------+-----------+
only showing top 10 rows


In [16]:
df_cleaned = df_cleaned.withColumn("event_timestamp", 
    F.to_timestamp(F.substring(F.col("event_time"), 1, 19), "yyyy-MM-dd HH:mm:ss"))

In [17]:
## Now let's look at our cleaned data
df_cleaned.show()

+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+
|          event_time|      event_type|product_id|        category_id|category_code|        brand|price|  user_id|        user_session| department|    event_timestamp|
+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+
|2019-10-01 00:00:...|            cart|   5773203|1487580005134238553|  accessories|       runail| 2.62|463240011|26dd6e6e-4dac-477...|accessories|2019-10-01 00:00:00|
|2019-10-01 00:00:...|            cart|   5773353|1487580005134238553|  accessories|       runail| 2.62|463240011|26dd6e6e-4dac-477...|accessories|2019-10-01 00:00:03|
|2019-10-01 00:00:...|            cart|   5881589|2151191071051219817|  accessories|       lovely|13.48|429681830|49e8d843-adf3-428...|accessories|2019-10-01 00

In [18]:
df_cleaned.describe().show()

+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+-------------------+--------------------+-----------+
|summary|          event_time|event_type|        product_id|         category_id|      category_code|        brand|             price|            user_id|        user_session| department|
+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+-------------------+--------------------+-----------+
|  count|             4096197|   4096197|           4096197|             4096197|            4096197|      4096197|           4096197|            4096197|             4095560|    4096197|
|   mean|                NULL|      NULL| 5467833.478498471|1.545586038596165...|               NULL|         NULL| 8.547736810511498| 5.01357513612451E8|                NULL|       NULL|
| stddev|                NULL|      NULL|1322743.0186315412|

In [19]:
# Fixing time zone setting
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [20]:
# Now let's create separate columns for date and time so later we can easily analize it
df_final = df_cleaned.select("*", 
                            F.to_date("event_time").alias("event_date"),
                            F.hour("event_time").alias("hour"))

In [21]:
df_final.show()

+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|          event_time|      event_type|product_id|        category_id|category_code|        brand|price|  user_id|        user_session| department|    event_timestamp|event_date|hour|
+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|2019-10-01 00:00:...|            cart|   5773203|1487580005134238553|  accessories|       runail| 2.62|463240011|26dd6e6e-4dac-477...|accessories|2019-10-01 05:00:00|2019-10-01|   0|
|2019-10-01 00:00:...|            cart|   5773353|1487580005134238553|  accessories|       runail| 2.62|463240011|26dd6e6e-4dac-477...|accessories|2019-10-01 05:00:03|2019-10-01|   0|
|2019-10-01 00:00:...|            cart|   5881589|2151191071051219817|  accessor

Above we can see that it shows 19 in Hour column even though in main event timestamp column it's 00:00. It's because our data is in UTC but when we use hour function it converts time into the local computer time which is currently Central Time Zone (CDT/CST). So basically 5 hours behind so if we can do the math (24-5) it will be 19

In [22]:
# Saving the file in parquet format
df_final.write.mode("overwrite").parquet("cleaned_data")

In [23]:
# Let's check
spark.read.parquet("cleaned_data").show()

+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|          event_time|      event_type|product_id|        category_id|category_code|        brand|price|  user_id|        user_session| department|    event_timestamp|event_date|hour|
+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|2019-10-18 12:08:...|            view|   5730223|1487580007457883075|  accessories|Unknown_brand| 4.44|468808792|cbec98c5-792b-4e9...|accessories|2019-10-18 17:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5695491|1487580013581566154|  accessories|        estel|  2.7|169486644|ee44d6fd-7b81-420...|accessories|2019-10-18 17:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5809870|1487580009387261981|  accessor